# BizInsight - Recommendation Evaluation and Testing

## Objective

Evaluate the correctness, relevance and reliability
of the generated business recommendations.

The evaluation verifies that recommendations are
supported by actual business data, correctly
prioritized and understandable to business owners.

## 1. Load Recommendation and Analysis Results

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

analysis_dir = project_dir / "results" / "business_analysis"

recommendation_dir = (
    project_dir / "results" / "recommendations"
)

recommendations_df = pd.read_csv(
    recommendation_dir / "recommendations.csv"
)

analysis_data = {}

for file in sorted(analysis_dir.glob("*.csv")):
    analysis_data[file.stem] = pd.read_csv(file)

print("Evaluation environment initialized successfully.")

print("Recommendations loaded:", len(recommendations_df))
print("Analysis datasets loaded:", len(analysis_data))

print("\nAvailable recommendation categories:")

display(recommendations_df["Category"].value_counts())

print("\nMissing values:")

display(recommendations_df.isnull().sum())

print("\nDuplicate recommendation IDs:",
      recommendations_df["RecommendationID"].duplicated().sum())

Evaluation environment initialized successfully.
Recommendations loaded: 736
Analysis datasets loaded: 12

Available recommendation categories:


Category
Product Returns          347
Product Profitability    339
Sales Anomalies           37
Store Performance          8
Promotion Performance      3
Business Performance       2
Name: count, dtype: int64


Missing values:


RecommendationID    0
Category            0
Type                0
Entity              0
Priority            0
Finding             0
Recommendation      0
dtype: int64


Duplicate recommendation IDs: 0


## 2. Recommendation Accuracy Validation

Verify that generated recommendations are supported
by the corresponding business analysis results.

Compare reported product sales, gross margins
and profitability against the original analysis data.

In [2]:
product_id = "PROD-100358"

print("Generated recommendation:\n")

display(
    recommendations_df[
        (recommendations_df["Entity"] == product_id)
        & (recommendations_df["Type"] == "Popular but Low Margin")
    ].style.set_properties(
        subset=["Finding", "Recommendation"],
        **{"white-space": "pre-wrap"}
    )
)

print("\nOriginal business analysis:\n")

display(
    analysis_data["product_performance"][
        analysis_data["product_performance"]["ProductID"]
        == product_id
    ].round(2)
)

Generated recommendation:



,RecommendationID,Category,Type,Entity,Priority,Finding,Recommendation
0,REC-0001,Product Profitability,Popular but Low Margin,PROD-100358,High,Product sold 263 net units with a gross margin of 22.95%.,Review the selling price and supplier costs. Consider testing a small price increase while monitoring sales volume.



Original business analysis:



,ProductID,NetSales,COGS,GrossProfit,NetUnitsSold,Records,GrossMargin
227,PROD-100358,20189.52,15555.44,4634.08,263.0,302,22.95


### Product Return Recommendation Validation

Verify that the generated return recommendations
accurately reflect the number of units sold,
returned units, and calculated return rates.

In [3]:
product_id = "PROD-168538"

print("Generated return recommendation:\n")

display(
    recommendations_df[
        (recommendations_df["Entity"] == product_id)
        & (recommendations_df["Type"] == "High Return Rate")
    ].style.set_properties(
        subset=["Finding", "Recommendation"],
        **{"white-space": "pre-wrap"}
    )
)

print("\nOriginal return analysis:\n")

display(
    analysis_data["return_analysis"][
        analysis_data["return_analysis"]["ProductID"]
        == product_id
    ].round(2)
)

Generated return recommendation:



,RecommendationID,Category,Type,Entity,Priority,Finding,Recommendation
160,REC-0161,Product Returns,High Return Rate,PROD-168538,High,"Product recorded 25 returned units out of 107 units sold, with a return rate of 23.36% and a total return value of 3,006.25.","Investigate the reasons for product returns. Review product quality, sizing accuracy, product descriptions and customer feedback. Consider discussing recurring issues with the supplier."



Original return analysis:



,ProductID,UnitsSold,UnitsReturned,ReturnValue,ReturnRate
1268,PROD-168538,107.0,25.0,3006.25,23.36


### Promotion Recommendation Validation

Verify that recommendations related to discount
strategies accurately reflect net sales, gross
profit and gross profit margins.

In [4]:
sales_type = "Clearance"

print("Generated promotion recommendation:\n")

display(
    recommendations_df[
        (recommendations_df["Entity"] == sales_type)
        & (recommendations_df["Category"] == "Promotion Performance")
    ].style.set_properties(
        subset=["Finding", "Recommendation"],
        **{"white-space": "pre-wrap"}
    )
)

print("\nOriginal promotion analysis:\n")

display(
    analysis_data["promotion_performance"][
        analysis_data["promotion_performance"]["SalesType"]
        == sales_type
    ].round(2)
)

Generated promotion recommendation:



,RecommendationID,Category,Type,Entity,Priority,Finding,Recommendation
159,REC-0160,Promotion Performance,Low Promotion Profitability,Clearance,High,"Generated net sales of 17,408.76 and gross profit of -13,681.57, with a gross margin of -78.59%.",Review the discount strategy and product costs. Consider reducing discount depth or limiting discounts to selected products.



Original promotion analysis:



,SalesType,NetSales,COGS,GrossProfit,NetUnitsSold,Records,GrossMargin,ProfitPerUnit
4,Clearance,17408.76,31090.33,-13681.57,1499.0,1353,-78.59,-9.13


### Store Performance Recommendation Validation

Verify that recommendations for underperforming
stores are supported by their financial performance
and average daily gross profit.

In [5]:
store_id = "STR-1308"

print("Generated store recommendation:\n")

display(
    recommendations_df[
        (recommendations_df["Entity"] == store_id)
        & (recommendations_df["Category"] == "Store Performance")
    ].style.set_properties(
        subset=["Finding", "Recommendation"],
        **{"white-space": "pre-wrap"}
    )
)

print("\nOriginal store analysis:\n")

display(
    analysis_data["store_performance"][
        analysis_data["store_performance"]["Store"]
        == store_id
    ].round(2)
)

Generated store recommendation:



,RecommendationID,Category,Type,Entity,Priority,Finding,Recommendation
720,REC-0721,Store Performance,Underperforming Store,STR-1308,Medium,"Average daily gross profit is 179.64, compared to the store median of 364.57.","Review store operations, product assortment and local customer demand. Compare performance with higher-performing stores."



Original store analysis:



,Store,NetSales,COGS,GrossProfit,NetUnitsSold,ActiveDays,GrossMargin,AverageDailySales,AverageDailyProfit
39,STR-1308,142256.04,87464.95,54791.09,2137.0,305,38.52,466.41,179.64


In [6]:
print(
    analysis_data["store_performance"]
    ["AverageDailyProfit"].median().round(2)
)

364.57


### Business Performance Recommendation Validation

Verify that recommendations related to declining
monthly sales are supported by historical financial
performance and monthly growth calculations.

In [7]:
month = "2025-09"

print("Generated business recommendation:\n")

display(
    recommendations_df[
        (recommendations_df["Entity"] == month)
        & (recommendations_df["Category"] == "Business Performance")
    ].style.set_properties(
        subset=["Finding", "Recommendation"],
        **{"white-space": "pre-wrap"}
    )
)

print("\nOriginal monthly performance:\n")

display(
    analysis_data["monthly_performance"][
        analysis_data["monthly_performance"]["Month"]
        .astype(str).isin(["2025-08", "2025-09"])
    ].round(2)
)

Generated business recommendation:



,RecommendationID,Category,Type,Entity,Priority,Finding,Recommendation
186,REC-0187,Business Performance,Declining Monthly Sales,2025-09,High,Monthly net sales decreased by 43.95% compared to the previous month.,"Investigate changes in customer demand, product availability and promotional activity. Consider testing targeted sales campaigns."



Original monthly performance:



,Month,NetSales,COGS,GrossProfit,NetUnitsSold,DaysRecorded,GrossMargin,CompleteMonth,SalesGrowth,ProfitGrowth
2,2025-08,1535117.21,831526.10,703591.11,17561.0,31,45.83,True,79.43,79.13
3,2025-09,860457.23,487868.96,372588.27,9654.0,30,43.30,True,-43.95,-47.04


### Sales Anomaly Recommendation Validation

Verify that recommendations related to unusual
sales patterns accurately reflect actual sales,
expected sales and percentage deviations.

In [8]:
anomaly_date = "2025-07-26"

print("Generated anomaly recommendation:\n")

display(
    recommendations_df[
        (recommendations_df["Entity"] == anomaly_date)
        & (recommendations_df["Category"] == "Sales Anomalies")
    ].style.set_properties(
        subset=["Finding", "Recommendation"],
        **{"white-space": "pre-wrap"}
    )
)

print("\nOriginal sales anomaly analysis:\n")

display(
    analysis_data["sales_anomalies"][
        analysis_data["sales_anomalies"]["Date"]
        .astype(str).str[:10] == anomaly_date
    ].round(2)
)

Generated anomaly recommendation:



,RecommendationID,Category,Type,Entity,Priority,Finding,Recommendation
721,REC-0722,Sales Anomalies,Unusually High Sales,2025-07-26,Medium,"Daily net sales were 94,090.95, compared to expected sales of 44,432.01. Sales deviation: +111.76%.",Investigate the factors behind unusually high sales. Consider whether successful promotions or demand patterns can be repeated.



Original sales anomaly analysis:



,Date,NetSales,GrossProfit,DayOfWeek,ExpectedSales,SalesDeviation,AnomalyType
3,2025-07-26,94090.95,44346.87,Saturday,44432.01,111.76,Unusually High Sales


## Recommendation Evaluation Summary

Representative recommendations from all six categories
were manually evaluated by comparing their findings
with the original business analysis results.

The evaluation considers numerical correctness,
relevance and practical usefulness.

In [10]:
evaluation_results = pd.DataFrame([
    ["Product Profitability", "REC-0001", True, True, True],
    ["Product Returns", "REC-0161", True, True, True],
    ["Promotion Performance", "REC-0160", True, True, True],
    ["Store Performance", "REC-0721", True, True, True],
    ["Business Performance", "REC-0187", True, True, True],
    ["Sales Anomalies", "REC-0722", True, True, True]
], columns=[
    "Category",
    "RecommendationID",
    "NumericalCorrectness",
    "Relevance",
    "Actionability"
])

criteria = [
    "NumericalCorrectness",
    "Relevance",
    "Actionability"
]

evaluation_results["Status"] = np.where(
    evaluation_results[criteria].all(axis=1),
    "PASS",
    "FAIL"
)

display(evaluation_results)

print("\nRecommendations evaluated:", len(evaluation_results))

print("Passed:",
      (evaluation_results["Status"] == "PASS").sum())

print("Sample pass rate:",
      evaluation_results["Status"].eq("PASS").mean() * 100,
      "%")

,Category,RecommendationID,NumericalCorrectness,Relevance,Actionability,Status
0,Product Profitability,REC-0001,True,True,True,PASS
1,Product Returns,REC-0161,True,True,True,PASS
2,Promotion Performance,REC-0160,True,True,True,PASS
3,Store Performance,REC-0721,True,True,True,PASS
4,Business Performance,REC-0187,True,True,True,PASS
5,Sales Anomalies,REC-0722,True,True,True,PASS



Recommendations evaluated: 6
Passed: 6
Sample pass rate: 100.0 %


## Save Recommendation Evaluation Results

The manual evaluation results are exported to a CSV
file for documentation, reporting and future testing.

In [11]:
evaluation_dir = project_dir / "results" / "evaluation"

evaluation_dir.mkdir(parents=True, exist_ok=True)

evaluation_results.to_csv(
    evaluation_dir / "recommendation_evaluation.csv",
    index=False
)

print("Recommendation evaluation saved successfully.")
print("Evaluated recommendations:", len(evaluation_results))
print("File:", evaluation_dir / "recommendation_evaluation.csv")

Recommendation evaluation saved successfully.
Evaluated recommendations: 6
File: /Users/martinagj/Desktop/BizInsight/bizinsight/results/evaluation/recommendation_evaluation.csv
